# 05c — Stable secondary-anchor independent evidence

Narrow audit of the 9 Stage-05b stable secondary anchors. This notebook does **not** relabel them as OFFICE.

In [ ]:
from pathlib import Path
import os, subprocess, sys, importlib.util
import pandas as pd

REPO_BRANCH = os.environ.get('GEOLIFE_REPO_BRANCH', 'main')
REPO_DIR = Path(os.environ.get('GEOLIFE_REPO_DIR', '/tmp/geolife'))
VOLUME_ROOT = Path('/mnt/geolife-data')
CACHE_05B = VOLUME_ROOT / 'cache' / 'cp2_v2' / '05b_home_consensus_adaptive_work'
CACHE_05C = VOLUME_ROOT / 'cache' / 'cp2_v2' / '05c_stable_secondary_evidence'
CACHE_05C.mkdir(parents=True, exist_ok=True)

if not (REPO_DIR / '.git').exists():
    subprocess.run(['git','clone','--depth','1','--branch',REPO_BRANCH,'https://github.com/tanh1c/geolife.git',str(REPO_DIR)], check=True)
else:
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin',REPO_BRANCH], check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'checkout',REPO_BRANCH], check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard',f'origin/{REPO_BRANCH}'], check=True)
for p in (REPO_DIR, REPO_DIR/'src'):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

try:
    from geolife.model import HomeOfficeConfig, build_semantic_locations
except ModuleNotFoundError:
    subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'], cwd=REPO_DIR, check=True)
    from geolife.model import HomeOfficeConfig, build_semantic_locations

spec = importlib.util.spec_from_file_location('stage05c', REPO_DIR/'analysis'/'05c_stable_secondary_independent_evidence.py')
stage05c = importlib.util.module_from_spec(spec)
spec.loader.exec_module(stage05c)
print(stage05c.synthetic_self_check())

In [ ]:
home_evidence = pd.read_pickle(CACHE_05B/'home_consensus_private.pkl')
work_patterns_42 = pd.read_pickle(CACHE_05B/'adaptive_work_patterns_42_private.pkl')
stable_cohort = stage05c.stable_secondary_cohort(work_patterns_42, home_evidence)
print('stable secondary users:', len(stable_cohort))
print('historical Beijing-v1 stable-secondary users:', stage05c.HISTORICAL_BEIJING_V1_STABLE_USERS)
print('CP2-v2 delta:', len(stable_cohort) - stage05c.HISTORICAL_BEIJING_V1_STABLE_USERS)


In [ ]:
def find_stay_cache():
    candidates = [
        VOLUME_ROOT/'cache'/'03a_user_behavior_deep_dive'/'stays_baseline_v1.pkl',
        VOLUME_ROOT/'artifacts'/'03a'/'stays_baseline_v1.pkl',
    ]
    for p in candidates:
        if p.exists(): return p
    matches = sorted(VOLUME_ROOT.glob('**/stays_baseline_v1.pkl'))
    if matches: return matches[0]
    raise FileNotFoundError('stays_baseline_v1.pkl')

stays = pd.read_pickle(find_stay_cache())
cfg = HomeOfficeConfig(clustering_method='complete_link', location_max_diameter_m=200.0)
semantic_stays, locations = build_semantic_locations(stays, config=cfg)
print('semantic users:', semantic_stays['user_id'].nunique())

In [ ]:
anchor_metrics = stage05c.build_anchor_metrics(semantic_stays, stable_cohort, min_active_days=3)
candidate_comparison = stage05c.rank_candidate_within_user(anchor_metrics)
metric_summary = stage05c.summarize_metric_comparison(candidate_comparison)
convergence_summary = stage05c.summarize_convergence(candidate_comparison)
bootstrap_summary = stage05c.paired_bootstrap_against_peer_median(candidate_comparison)
support_sensitivity = stage05c.support_sensitivity(semantic_stays, stable_cohort)
static_comparison = stage05c.compare_to_static_office(candidate_comparison, work_patterns_42)

display(metric_summary)
display(convergence_summary)
display(bootstrap_summary)
display(support_sensitivity)
display(static_comparison)

In [ ]:
snapshot = pd.DataFrame([{
    'stable_secondary_users': len(stable_cohort),
    'users_with_fair_comparator': candidate_comparison['user_id'].nunique(),
    'three_plus_top_axes_users': int(candidate_comparison['top1_evidence_axes'].ge(3).sum()),
    'two_top_axes_users': int(candidate_comparison['top1_evidence_axes'].eq(2).sum()),
    'zero_or_one_top_axis_users': int(candidate_comparison['top1_evidence_axes'].le(1).sum()),
}])
display(snapshot)

anchor_metrics.to_pickle(CACHE_05C/'anchor_metrics_private.pkl')
candidate_comparison.to_pickle(CACHE_05C/'candidate_comparison_private.pkl')
metric_summary.to_csv(CACHE_05C/'metric_summary.csv', index=False)
convergence_summary.to_csv(CACHE_05C/'convergence_summary.csv', index=False)
bootstrap_summary.to_csv(CACHE_05C/'paired_bootstrap_summary.csv', index=False)
support_sensitivity.to_csv(CACHE_05C/'support_sensitivity.csv', index=False)
static_comparison.to_csv(CACHE_05C/'static_office_stratified_evidence.csv', index=False)
snapshot.to_csv(CACHE_05C/'decision_snapshot.csv', index=False)

## Interpretation

A stable secondary anchor is behavioral evidence only. If independent axes remain mixed, keep `stable_secondary_anchor` descriptive and do not expand semantic WORK/OFFICE.